# Partie 4 - Etape 3: Strategie 3 Teacher-Student (Noisy Student)

Cette experience adapte l'idee de **Noisy Student** (Xie et al., 2020) au cadre multi-fidelite :
- un **Teacher** entraine sur HF uniquement,
- generation de **soft pseudo-labels** sur BF,
- entrainement d'un **Student noised** sur HF (labels reels) + BF (pseudo-labels).

## Objectif
- Prioriser la performance sur Test HF.
- Exploiter le volume BF avec pseudo-labels sans cout d'annotation additionnel.
- Evaluer HF / BF / Mixte et le cout total en CA.

In [ ]:
import os
import json
import time
import random
import shutil

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, Subset, ConcatDataset
from torchvision import datasets, transforms, models
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil: {device}')

# --- Module de degradation partage (source unique de la degradation BF) ---
import sys
import importlib
_SRC_CANDIDATES = [
    '/content/drive/MyDrive/UTBM_PF22/src',  # Colab (src synchronise sur Drive)
    '../src', 'src', './src',                # execution locale (depuis notebooks/ ou racine)
]
for _p in _SRC_CANDIDATES:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import degradation
importlib.reload(degradation)
from degradation import (
    DegradedDataset, hf_transform, clean_tensor_transform, normalize_transform,
    IMAGENET_MEAN, IMAGENET_STD,
)
print('Module degradation charge depuis:', os.path.dirname(degradation.__file__))
import cost
importlib.reload(cost)
from cost import data_cost, unit_cost
import env_config
importlib.reload(env_config)

## 1) Montage Drive et chemins (continuite notebooks precedents)
SSD local prioritaire, extraction ZIP si necessaire, fallback Drive.

In [ ]:
DATASET_NAME = "Animals-10"
BASE_DIR = env_config.ensure_dataset_ready(DATASET_NAME)
RESULTS_DIR = env_config.results_dir(DATASET_NAME)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Dataset    :', DATASET_NAME)
print('BASE_DIR   :', BASE_DIR)
print('RESULTS_DIR:', RESULTS_DIR)

## 2) Configuration experimentale et transforms

In [ ]:
COST_HF = 10
COST_BF = 1

NUM_CLASSES = 10
BATCH_SIZE = 64
NUM_WORKERS = min(8, (os.cpu_count() or 2))  # Colab ~2, serveur 16 vCPU

TEACHER_EPOCHS = 10
STUDENT_EPOCHS = 20
TEACHER_LR = 1e-3
STUDENT_LR = 1e-3
HF_TRAIN_RATIO = 0.8
ALPHA_BF = 1.0
STUDENT_DROPOUT = 0.3
TEMPERATURE = 1.0

# --- Degradation BF a la volee via le module partage src/degradation.py ---
# Les images BF sont stockees propres ; la degradation canonique est appliquee
# a la volee (coherente avec le test). Pour le Student "noised", l'augmentation
# agressive (RandAugment, ColorJitter, RandomErasing) est appliquee APRES la
# degradation BF, via le post_transform de DegradedDataset.

# Transforms HF "propres" (teacher, validation, tests)
transform_hf = hf_transform()              # Resize -> ToTensor -> Normalize
transform_clean = clean_tensor_transform() # Resize -> ToTensor (pour DegradedDataset)

# Augmentation agressive du Student appliquee a des images HF PROPRES (PIL)
transform_student_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.08),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.35, scale=(0.02, 0.25), ratio=(0.3, 3.3), value='random'),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Augmentation agressive du Student appliquee APRES la degradation BF.
# Entree = tenseur degrade [0,1] -> retour en PIL pour RandAugment/ColorJitter,
# puis ToTensor + RandomErasing + Normalize.
student_post_bf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.08),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.35, scale=(0.02, 0.25), ratio=(0.3, 3.3), value='random'),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

required_dirs = [
    os.path.join(BASE_DIR, 'train', 'HF'),
    os.path.join(BASE_DIR, 'train', 'BF'),
    os.path.join(BASE_DIR, 'test'),
]
missing = [p for p in required_dirs if not os.path.isdir(p)]
if missing:
    raise FileNotFoundError('Structure dataset invalide. Dossiers manquants: ' + ' | '.join(missing))

print(f'NUM_WORKERS utilise: {NUM_WORKERS}')

## 3) Construction des datasets/dataloaders

In [ ]:
def make_imagefolder(path, transform):
    return datasets.ImageFolder(path, transform=transform)

# Datasets de base
hf_full_std = make_imagefolder(os.path.join(BASE_DIR, 'train/HF'), transform_hf)
# BF "vues comme a l'evaluation" (degradation canonique deterministe) pour le pseudo-labeling
bf_full_std = DegradedDataset(
    make_imagefolder(os.path.join(BASE_DIR, 'train/BF'), transform_clean),
    seeded=True,
)
test_hf = make_imagefolder(os.path.join(BASE_DIR, 'test'), transform_hf)
test_bf = DegradedDataset(
    make_imagefolder(os.path.join(BASE_DIR, 'test'), transform_clean),
    seeded=True,
)

hf_train_size = int(HF_TRAIN_RATIO * len(hf_full_std))
hf_val_size = len(hf_full_std) - hf_train_size
hf_train_subset, hf_val_subset = random_split(
    hf_full_std,
    [hf_train_size, hf_val_size],
    generator=torch.Generator().manual_seed(SEED)
)

hf_train_indices = hf_train_subset.indices
hf_val_indices = hf_val_subset.indices

# Versions transformees pour le Student (agressif)
hf_full_student = make_imagefolder(os.path.join(BASE_DIR, 'train/HF'), transform_student_train)
# BF Student = degradation canonique (deterministe par index) PUIS augmentation agressive.
# Aligne par index avec bf_full_std -> les pseudo-labels correspondent aux memes images.
bf_full_student = DegradedDataset(
    make_imagefolder(os.path.join(BASE_DIR, 'train/BF'), transform_clean),
    post_transform=student_post_bf,
    seeded=True,
)

hf_train_student_subset = Subset(hf_full_student, hf_train_indices)

# Dataloaders teacher/validation/tests
loader_hf_teacher_train = DataLoader(hf_train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
loader_hf_val = DataLoader(hf_val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
loader_bf_for_pseudo = DataLoader(bf_full_std, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
loader_test_hf = DataLoader(test_hf, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
loader_test_bf = DataLoader(test_bf, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'HF train: {len(hf_train_subset)} | HF val: {len(hf_val_subset)}')
print(f'BF train (pseudo): {len(bf_full_std)} | Test HF/BF: {len(test_hf)}')

## 4) Utilitaires modeles, AMP et evaluation

In [6]:
def create_resnet18(num_classes=10, dropout_p=0.0):
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    if dropout_p > 0:
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(in_features, num_classes)
        )
    else:
        model.fc = nn.Linear(in_features, num_classes)
    return model.to(device)

def make_grad_scaler():
    try:
        return GradScaler('cuda', enabled=(device.type == 'cuda'))
    except TypeError:
        return GradScaler(enabled=(device.type == 'cuda'))

@torch.no_grad()
def evaluate_hard_labels(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            logits = model(x)
            loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += x.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples

def soft_cross_entropy(logits, soft_targets, temperature=1.0):
    log_probs = F.log_softmax(logits / temperature, dim=1)
    return -(soft_targets * log_probs).sum(dim=1).mean()

## 5) Etape 1: entrainement du Teacher sur HF uniquement

In [7]:
def train_teacher_on_hf(model, train_loader, val_loader, epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler = make_grad_scaler()

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        epoch_start = time.time()

        for batch_idx, (x, y) in enumerate(train_loader, start=1):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)
            seen += x.size(0)

            if batch_idx == 1 or batch_idx % 10 == 0 or batch_idx == len(train_loader):
                elapsed = time.time() - epoch_start
                bps = batch_idx / elapsed if elapsed > 0 else 0.0
                eta_sec = (len(train_loader) - batch_idx) / bps if bps > 0 else 0.0
                print(
                    f'[Teacher] Epoch {epoch}/{epochs} | Batch {batch_idx}/{len(train_loader)} | '
                    f'loss={loss.item():.4f} | {bps:.2f} batch/s | ETA={eta_sec/60:.1f} min'
                )

        train_loss = running_loss / max(1, seen)
        val_loss, val_acc = evaluate_hard_labels(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f'[Teacher] Epoch {epoch}/{epochs} terminee | '
            f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.2f}%'
        )

    total_min = (time.time() - start_time) / 60.0
    print(f'[Teacher] Entrainement termine en {total_min:.2f} min')
    return history, total_min

teacher = create_resnet18(num_classes=NUM_CLASSES, dropout_p=0.0)
print('Teacher initialise:', teacher.__class__.__name__)
teacher_hist, teacher_time_min = train_teacher_on_hf(
    model=teacher,
    train_loader=loader_hf_teacher_train,
    val_loader=loader_hf_val,
    epochs=TEACHER_EPOCHS,
    lr=TEACHER_LR
)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print('Teacher fige en mode eval (aucune mise a jour gradient).')

Teacher initialise: ResNet
[Teacher] Epoch 1/10 | Batch 1/30 | loss=2.4752 | 0.06 batch/s | ETA=8.2 min
[Teacher] Epoch 1/10 | Batch 10/30 | loss=2.3694 | 0.57 batch/s | ETA=0.6 min
[Teacher] Epoch 1/10 | Batch 20/30 | loss=2.0788 | 1.07 batch/s | ETA=0.2 min
[Teacher] Epoch 1/10 | Batch 30/30 | loss=1.9957 | 0.91 batch/s | ETA=0.0 min
[Teacher] Epoch 1/10 terminee | train_loss=2.2782 | val_loss=4.1685 | val_acc=11.04%
[Teacher] Epoch 2/10 | Batch 1/30 | loss=1.8155 | 2.91 batch/s | ETA=0.2 min
[Teacher] Epoch 2/10 | Batch 10/30 | loss=1.9441 | 7.36 batch/s | ETA=0.0 min
[Teacher] Epoch 2/10 | Batch 20/30 | loss=1.7669 | 8.21 batch/s | ETA=0.0 min
[Teacher] Epoch 2/10 | Batch 30/30 | loss=1.8328 | 9.24 batch/s | ETA=0.0 min
[Teacher] Epoch 2/10 terminee | train_loss=1.9320 | val_loss=2.2541 | val_acc=25.27%
[Teacher] Epoch 3/10 | Batch 1/30 | loss=1.6831 | 2.99 batch/s | ETA=0.2 min
[Teacher] Epoch 3/10 | Batch 10/30 | loss=1.6951 | 8.63 batch/s | ETA=0.0 min
[Teacher] Epoch 3/10 | Bat

## 6) Etape 2: pseudo-labeling soft sur BF

In [8]:
@torch.no_grad()
def generate_soft_pseudo_labels(teacher_model, bf_loader, temperature=1.0):
    teacher_model.eval()
    probs_all = []

    for batch_idx, (x, _) in enumerate(bf_loader, start=1):
        x = x.to(device)
        with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            logits = teacher_model(x)
            probs = torch.softmax(logits / temperature, dim=1)

        probs_all.append(probs.detach().cpu())

        if batch_idx == 1 or batch_idx % 20 == 0 or batch_idx == len(bf_loader):
            print(f'[Pseudo] Batch {batch_idx}/{len(bf_loader)} traite')

    return torch.cat(probs_all, dim=0)

soft_pseudo_labels_bf = generate_soft_pseudo_labels(
    teacher_model=teacher,
    bf_loader=loader_bf_for_pseudo,
    temperature=TEMPERATURE
)

print('Pseudo-labels BF generes:', tuple(soft_pseudo_labels_bf.shape))

[Pseudo] Batch 1/332 traite
[Pseudo] Batch 20/332 traite
[Pseudo] Batch 40/332 traite
[Pseudo] Batch 60/332 traite
[Pseudo] Batch 80/332 traite
[Pseudo] Batch 100/332 traite
[Pseudo] Batch 120/332 traite
[Pseudo] Batch 140/332 traite
[Pseudo] Batch 160/332 traite
[Pseudo] Batch 180/332 traite
[Pseudo] Batch 200/332 traite
[Pseudo] Batch 220/332 traite
[Pseudo] Batch 240/332 traite
[Pseudo] Batch 260/332 traite
[Pseudo] Batch 280/332 traite
[Pseudo] Batch 300/332 traite
[Pseudo] Batch 320/332 traite
[Pseudo] Batch 332/332 traite
Pseudo-labels BF generes: (21213, 10)


## 7) Datasets Student (HF hard labels + BF soft pseudo-labels)

In [9]:
class HFHardLabelDataset(Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        # hard_label = etiquette reelle HF, soft_label vide
        soft_label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        domain = 1  # 1 = HF
        return x, y, soft_label, domain

class BFPseudoSoftDataset(Dataset):
    def __init__(self, bf_dataset_student, pseudo_probs):
        self.bf_dataset_student = bf_dataset_student
        self.pseudo_probs = pseudo_probs.float()
        if len(self.bf_dataset_student) != len(self.pseudo_probs):
            raise ValueError('Mismatch taille BF dataset vs pseudo-labels')

    def __len__(self):
        return len(self.bf_dataset_student)

    def __getitem__(self, idx):
        x, _ = self.bf_dataset_student[idx]
        hard_label_dummy = -1
        soft_label = self.pseudo_probs[idx]
        domain = 0  # 0 = BF pseudo
        return x, hard_label_dummy, soft_label, domain

hf_student_ds = HFHardLabelDataset(hf_train_student_subset)
bf_student_ds = BFPseudoSoftDataset(bf_full_student, soft_pseudo_labels_bf)

student_train_ds = ConcatDataset([hf_student_ds, bf_student_ds])
loader_student_train = DataLoader(
    student_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f'Student HF samples: {len(hf_student_ds)}')
print(f'Student BF samples: {len(bf_student_ds)}')
print(f'Student total train samples: {len(student_train_ds)} | batches/epoch: {len(loader_student_train)}')

Student HF samples: 1881
Student BF samples: 21213
Student total train samples: 23094 | batches/epoch: 361


## 8) Etape 4: entrainement Student noised
Loss totale = Loss_HF + alpha * Loss_BF
- HF: CrossEntropyLoss standard sur hard labels
- BF: Soft Cross-Entropy sur pseudo-labels

In [10]:
def train_student_mixed(student_model, train_loader, val_loader, epochs, lr, alpha_bf=1.0):
    ce_hard = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student_model.parameters(), lr=lr)
    scaler = make_grad_scaler()

    history = {
        'train_total_loss': [],
        'train_hf_loss': [],
        'train_bf_loss': [],
        'val_loss': [],
        'val_acc': []
    }

    start_time = time.time()
    print(f'Alpha BF utilise: {alpha_bf}')

    for epoch in range(1, epochs + 1):
        student_model.train()
        epoch_start = time.time()

        total_running = 0.0
        hf_running = 0.0
        bf_running = 0.0
        total_steps = 0
        hf_steps = 0
        bf_steps = 0

        for batch_idx, (x, y_hard, y_soft, domain) in enumerate(train_loader, start=1):
            x = x.to(device, non_blocking=True)
            y_hard = y_hard.to(device, non_blocking=True)
            y_soft = y_soft.to(device, non_blocking=True)
            domain = domain.to(device, non_blocking=True)

            hf_mask = (domain == 1)
            bf_mask = (domain == 0)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = student_model(x)

                if hf_mask.any():
                    hf_loss = ce_hard(logits[hf_mask], y_hard[hf_mask])
                else:
                    hf_loss = torch.zeros((), device=device)

                if bf_mask.any():
                    bf_loss = soft_cross_entropy(
                        logits=logits[bf_mask],
                        soft_targets=y_soft[bf_mask],
                        temperature=TEMPERATURE
                    )
                else:
                    bf_loss = torch.zeros((), device=device)

                total_loss = hf_loss + alpha_bf * bf_loss

            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_running += total_loss.item()
            total_steps += 1

            if hf_mask.any():
                hf_running += hf_loss.item()
                hf_steps += 1
            if bf_mask.any():
                bf_running += bf_loss.item()
                bf_steps += 1

            if batch_idx == 1 or batch_idx % 20 == 0 or batch_idx == len(train_loader):
                elapsed = time.time() - epoch_start
                bps = batch_idx / elapsed if elapsed > 0 else 0.0
                eta_sec = (len(train_loader) - batch_idx) / bps if bps > 0 else 0.0
                print(
                    f'[Student] Epoch {epoch}/{epochs} | Batch {batch_idx}/{len(train_loader)} | '
                    f'total={total_loss.item():.4f} | hf={hf_loss.item():.4f} | bf={bf_loss.item():.4f} | '
                    f'{bps:.2f} batch/s | ETA={eta_sec/60:.1f} min'
                )

        train_total = total_running / max(1, total_steps)
        train_hf = hf_running / max(1, hf_steps)
        train_bf = bf_running / max(1, bf_steps)

        val_loss, val_acc = evaluate_hard_labels(student_model, val_loader, ce_hard)

        history['train_total_loss'].append(train_total)
        history['train_hf_loss'].append(train_hf)
        history['train_bf_loss'].append(train_bf)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f'[Student] Epoch {epoch}/{epochs} terminee | '
            f'total={train_total:.4f} | hf={train_hf:.4f} | bf={train_bf:.4f} | '
            f'val_loss={val_loss:.4f} | val_acc={val_acc:.2f}%'
        )

    total_min = (time.time() - start_time) / 60.0
    print(f'[Student] Entrainement termine en {total_min:.2f} min')
    return history, total_min

student = create_resnet18(num_classes=NUM_CLASSES, dropout_p=STUDENT_DROPOUT)
print('Student initialise:', student.__class__.__name__)

student_hist, student_time_min = train_student_mixed(
    student_model=student,
    train_loader=loader_student_train,
    val_loader=loader_hf_val,
    epochs=STUDENT_EPOCHS,
    lr=STUDENT_LR,
    alpha_bf=ALPHA_BF
)

Student initialise: ResNet
Alpha BF utilise: 1.0
[Student] Epoch 1/20 | Batch 1/361 | total=4.7526 | hf=2.2295 | bf=2.5231 | 0.86 batch/s | ETA=7.0 min
[Student] Epoch 1/20 | Batch 20/361 | total=4.7269 | hf=2.5203 | bf=2.2066 | 3.07 batch/s | ETA=1.8 min
[Student] Epoch 1/20 | Batch 40/361 | total=4.4807 | hf=2.3362 | bf=2.1445 | 2.96 batch/s | ETA=1.8 min
[Student] Epoch 1/20 | Batch 60/361 | total=5.1908 | hf=3.0475 | bf=2.1433 | 3.04 batch/s | ETA=1.6 min
[Student] Epoch 1/20 | Batch 80/361 | total=5.0226 | hf=2.7324 | bf=2.2902 | 3.07 batch/s | ETA=1.5 min
[Student] Epoch 1/20 | Batch 100/361 | total=3.8745 | hf=1.7100 | bf=2.1646 | 3.06 batch/s | ETA=1.4 min
[Student] Epoch 1/20 | Batch 120/361 | total=4.5767 | hf=2.4782 | bf=2.0985 | 3.08 batch/s | ETA=1.3 min
[Student] Epoch 1/20 | Batch 140/361 | total=4.3232 | hf=2.1762 | bf=2.1470 | 3.08 batch/s | ETA=1.2 min
[Student] Epoch 1/20 | Batch 160/361 | total=4.1586 | hf=2.1489 | bf=2.0097 | 3.06 batch/s | ETA=1.1 min
[Student] Ep

## 9) Etape 5: cout CA, evaluation finale et sauvegardes

In [ ]:
criterion = nn.CrossEntropyLoss()
test_hf_loss, test_hf_acc = evaluate_hard_labels(student, loader_test_hf, criterion)
test_bf_loss, test_bf_acc = evaluate_hard_labels(student, loader_test_bf, criterion)
# Test Mixte = jeu mixte equilibre : chaque image de test est evaluee une fois
# en HF ET une fois en BF (2N predictions). Moyenner les deux accuracies (memes
# N images) equivaut EXACTEMENT a l'accuracy sur ce jeu mixte (deterministe).
test_mix_acc = (test_hf_acc + test_bf_acc) / 2.0

# Cout CA: teacher(HF) + student(HF+BF). Pseudo-labeling = 0 CA.
# Cout DONNEE (acquisition, unique) : pools HF + BF comptes une seule fois
# (le pseudo-labeling n'ajoute aucun cout d'annotation).
data_cost_CA = data_cost(n_hf=len(hf_full_std), n_bf=len(bf_full_std))
# Cout TOTAL (pondere par le cout unitaire = ancien cout) : teacher + student.
total_cost_ca = (len(hf_train_subset) * COST_HF * TEACHER_EPOCHS
                 + (len(hf_train_subset) * COST_HF + len(bf_full_std) * COST_BF) * STUDENT_EPOCHS)
# Cout CALCUL (images vues) : teacher (HF) + student (HF + BF pseudo).
compute_images_seen = (len(hf_train_subset) * TEACHER_EPOCHS
                       + len(student_train_ds) * STUDENT_EPOCHS)
# Decomposition par domaine (pour la sensibilite au ratio HF:BF).
n_hf_pool = len(hf_full_std)
n_bf_pool = len(bf_full_std)
hf_images_seen = len(hf_train_subset) * TEACHER_EPOCHS + len(hf_train_subset) * STUDENT_EPOCHS
bf_images_seen = len(bf_full_std) * STUDENT_EPOCHS

results = {
    'strategy': 'teacher_student_noisy_student',
    'paper_reference': 'Xie et al. 2020',
    'teacher_epochs': TEACHER_EPOCHS,
    'student_epochs': STUDENT_EPOCHS,
    'teacher_lr': TEACHER_LR,
    'student_lr': STUDENT_LR,
    'alpha_bf': ALPHA_BF,
    'temperature': TEMPERATURE,
    'student_dropout': STUDENT_DROPOUT,
    'data_cost_CA': float(data_cost_CA),
    'compute_images_seen': int(compute_images_seen),
    'total_cost_CA': float(total_cost_ca),
    'n_hf_pool': int(n_hf_pool), 'n_bf_pool': int(n_bf_pool),
    'hf_images_seen': int(hf_images_seen), 'bf_images_seen': int(bf_images_seen),
    'teacher_time_min': float(teacher_time_min),
    'student_time_min': float(student_time_min),
    'total_time_min': float(teacher_time_min + student_time_min),
    'test_hf_loss': float(test_hf_loss),
    'test_bf_loss': float(test_bf_loss),
    'accuracy_HF': float(test_hf_acc),
    'accuracy_BF': float(test_bf_acc),
    'accuracy_Mixte': float(test_mix_acc)
}

json_path = os.path.join(RESULTS_DIR, 'results_strategy3_noisy_student.json')
teacher_pth_path = os.path.join(RESULTS_DIR, 'model_strategy3_teacher.pth')
student_pth_path = os.path.join(RESULTS_DIR, 'model_strategy3_student.pth')
png_path = os.path.join(RESULTS_DIR, 'strategy3_noisy_student_curves.png')

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

torch.save(teacher.state_dict(), teacher_pth_path)
torch.save(student.state_dict(), student_pth_path)

print('--- RESULTATS FINAUX STRATEGIE 3 ---')
print(f'Test HF    : {test_hf_acc:.2f}%')
print(f'Test BF    : {test_bf_acc:.2f}%')
print(f'Test Mixte : {test_mix_acc:.2f}%')
print(f'Cout total : {total_cost_ca} CA')
print(f'Temps total: {teacher_time_min + student_time_min:.2f} min')
print('JSON sauve       :', json_path)
print('Teacher model    :', teacher_pth_path)
print('Student model    :', student_pth_path)

x_t = np.arange(1, TEACHER_EPOCHS + 1)
x_s = np.arange(1, STUDENT_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_t, teacher_hist['train_loss'], marker='o', label='Teacher train loss (HF)')
axes[0].plot(x_t, teacher_hist['val_loss'], marker='o', linestyle='--', label='Teacher val loss (HF)')
axes[0].set_title('Teacher (HF only)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].plot(x_s, student_hist['train_total_loss'], marker='o', label='Student total loss')
axes[1].plot(x_s, student_hist['train_hf_loss'], marker='o', alpha=0.75, label='Student HF hard loss')
axes[1].plot(x_s, student_hist['train_bf_loss'], marker='o', alpha=0.75, label='Student BF soft loss')
axes[1].plot(x_s, student_hist['val_loss'], marker='o', linestyle='--', label='Student val loss HF')
axes[1].set_title('Student noised (HF hard + BF pseudo-soft)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(png_path, dpi=180, bbox_inches='tight')
plt.show()
print('Figure sauvee:', png_path)

## 10) Comparaison optionnelle avec baselines + strategies 1 et 2

In [12]:
comparison_files = {
    'BL 1(HF)': 'results_baseline_HF.json',
    'BL 2(BF)': 'results_baseline_BF.json',
    'BL 3(MIXTE)': 'results_baseline_MIXTE.json',
    'Strategie 1(BF->HF)': 'results_strategy1_transfer_learning.json',
    'Strategie 2(CoTrain)': 'results_strategy2_cotraining_reweighting.json',
    'Strategie 3(NoisyStudent)': 'results_strategy3_noisy_student.json'
}

rows = []
for name, fn in comparison_files.items():
    p = os.path.join(RESULTS_DIR, fn)
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            d = json.load(f)
        rows.append((
            name,
            d.get('accuracy_HF', np.nan),
            d.get('accuracy_BF', np.nan),
            d.get('accuracy_Mixte', np.nan),
            d.get('total_cost_CA', np.nan),
            d.get('total_time_min', d.get('training_time_min', np.nan))
        ))

if not rows:
    print('Aucun fichier JSON trouve pour la comparaison.')
else:
    header = f"{'Modele':<30} {'HF%':>8} {'BF%':>8} {'Mixte%':>10} {'Cout(CA)':>12} {'Temps(min)':>12}"
    print(header)
    print('-' * len(header))
    for r in rows:
        print(f"{r[0]:<30} {r[1]:>8.2f} {r[2]:>8.2f} {r[3]:>10.2f} {r[4]:>12.0f} {r[5]:>12.2f}")

Modele                              HF%      BF%     Mixte%     Cout(CA)   Temps(min)
-------------------------------------------------------------------------------------
BL 1(HF)                          56.20    35.16      45.68       470400          nan
BL 2(BF)                          59.07    63.66      61.36       424260          nan
BL 3(MIXTE)                       75.90    65.03      70.47       894660          nan
Strategie 1(BF->HF)               78.84    57.65      68.25       400230         7.83
Strategie 2(CoTrain)              70.12    61.90      66.01       486720         5.10
Strategie 3(NoisyStudent)         52.72    28.46      40.59       988560        40.85
